# SVM Taxi Demand Prediction — Community Area, 1-Hour Resolution

**Same pooled-SVM approach as `03.01`, at a finer time resolution (1h vs. 4h).**

- Same 77 community areas; 1h windows means ~4x the rows and heavier zero-inflation.
- **Why this notebook exists**: a genuine scaling stress test for SVM training cost, not
  just a resolution swap.
- Mirrors `03.01` section-for-section; only what's new or different at 1h is called out
  here -- see `03.01` for the underlying reasoning (log-scale target, skill score, why
  pooled, validation strategy).
- **Course constraint**: same as `03.01` -- no lagged/autoregressive demand features.

**Notebook structure**:
1. Data loading (1) & panel construction (2) & features (3)
2. Profile baseline (4)
3. Pooled `LinearSVR` -- no kernel, feature importance (5)
4. Kernel comparison & hyperparameter tuning (6-7)
5. Results -- model comparison, per-area performance (8)
6. Export results (9)
7. Shortfalls & honest limitations (10)
8. Outlook (11)

Demand tiering and per-area models (`03.01`'s section 9) are not repeated here -- both were
tried and dropped at this resolution; see section 10 for why.

## Validation Strategy

24 hourly profile-baseline buckets instead of 4h's 6 -- a finer, harder-to-beat reference
at this resolution. Everything else (holdout split, `KFold` tuning, skill score) is
unchanged from `03.01`.

In [ ]:
import time
import warnings
import pathlib

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import contextily as ctx
import holidays

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TAXI_DATA_PATH       = "../../data/chicago_taxi_2024_2026_clean.parquet"
WEATHER_DATA_PATH    = "../../data/chicago_weather_2024_2026_hourly.csv"
COMMUNITY_AREAS_PATH = "../../data/community_areas_chicago.geojson"

# Data range covered by the cleaned parquet (June 2026+ excluded: portal reporting lag)
DATA_START    = pd.Timestamp("2024-01-01")
DATA_END_EXCL = pd.Timestamp("2026-06-01")

# Chronological split: train 2024-2025 (two full years), test Jan-May 2026
TRAIN_END = pd.Timestamp("2026-01-01")

# Evaluation week for baseline comparison
EVAL_WEEK_START = pd.Timestamp("2026-03-02")
EVAL_WEEK_END   = EVAL_WEEK_START + pd.Timedelta(days=7)

FREQ = "1h"  # community-area resolution at 1h windows -- see 03.01 for the 4h version
WEATHER_COLS = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "cloud_cover"]

ALL_AREAS = np.arange(1, 78)  # official community areas 1..77 -- ALL of them, no threshold

# expm1(709) overflows float64; clip the log-scale prediction first so an unstable kernel
# (see the kernel comparison below) gets a very bad but finite score instead of crashing.
LOG_PRED_CAP = 20

# Chart colors (validated palette: sequential blue, diverging blue<->red)
SEQUENTIAL_BLUE = "#256abf"
DIVERGING_CMAP = LinearSegmentedColormap.from_list("diverging_blue_red", ["#e34948", "#f0efec", "#2a78d6"])
# Categorical palette -- only needed where >2 series share one axes (the multi-model overlay
# in section 8.2); same palette as 03.05's NN notebook for visual consistency across notebooks.
CATEGORICAL_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]

# Single seed used everywhere randomness enters (subsampling, model random_state, CV) so a
# full rerun is reproducible end to end.
SEED = 42
np.random.seed(SEED)

# LinearSVR (liblinear) at some C values doesn't reach its optimum within max_iter on this
# data -- the fits are still valid (just not at the exact optimum), and raising max_iter
# further would not remove the warning either (liblinear's tolerance is effectively
# unreachable for these C/data combinations). Silenced rather than chasing an unreachable
# tolerance across hundreds of grid-search fits.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

print(f"train: {DATA_START.date()} to {TRAIN_END.date()}  "
      f"test: {TRAIN_END.date()} to {DATA_END_EXCL.date()}")

: 

: 

## 1. Data Loading

In [ ]:
taxi = pd.read_parquet(TAXI_DATA_PATH, columns=["trip_start_timestamp", "pickup_community_area"])
n_raw = len(taxi)
taxi = taxi[taxi["pickup_community_area"].notna()].copy()
taxi["pickup_community_area"] = taxi["pickup_community_area"].astype(int)
print(f"Trips with pickup CA: {len(taxi):,} of {n_raw:,} ({len(taxi)/n_raw*100:.1f}%)")

weather = pd.read_csv(WEATHER_DATA_PATH)
weather["datetime"] = pd.to_datetime(weather["datetime"])
weather = weather[["datetime"] + WEATHER_COLS]
print(f"Weather rows: {len(weather):,}")

## 2. Building the Pooled Demand Panel

Now at 1h windows -- ~4x the rows and a higher zero-demand share than 4h, since a single
hour is much more likely to see no pickups than a 4h block.

In [ ]:
# US federal holidays (incl. observed dates), computed for every year the data covers --
# avoids maintaining a hand-picked date list that silently goes stale outside 2024-2026.
US_HOLIDAYS = pd.DatetimeIndex(sorted(
    holidays.US(years=range(DATA_START.year, DATA_END_EXCL.year + 1))
))

window = taxi["trip_start_timestamp"].dt.floor(FREQ).rename("timestamp")
counts = taxi.groupby(["pickup_community_area", window]).size().rename("count")

all_windows = pd.date_range(DATA_START, DATA_END_EXCL, freq=FREQ, inclusive="left")
grid = pd.MultiIndex.from_product([ALL_AREAS, all_windows], names=["community_area", "timestamp"])
panel = counts.reindex(grid, fill_value=0).reset_index()

w = weather.copy()
w["timestamp"] = w["datetime"].dt.floor(FREQ)
w_agg = w.groupby("timestamp")[WEATHER_COLS].mean().reset_index()
panel = panel.merge(w_agg, on="timestamp", how="left")

panel["hour"]        = panel["timestamp"].dt.hour
panel["day_of_week"] = panel["timestamp"].dt.dayofweek
panel["month"]       = panel["timestamp"].dt.month

panel["hour_sin"]         = np.sin(2 * np.pi * panel["hour"] / 24)
panel["hour_cos"]         = np.cos(2 * np.pi * panel["hour"] / 24)
panel["hour_sin2"]        = np.sin(4 * np.pi * panel["hour"] / 24)   # 2nd harmonic (captures bimodal rush pattern)
panel["hour_cos2"]        = np.cos(4 * np.pi * panel["hour"] / 24)
panel["day_of_week_sin"]  = np.sin(2 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_cos"]  = np.cos(2 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_sin2"] = np.sin(4 * np.pi * panel["day_of_week"] / 7)
panel["day_of_week_cos2"] = np.cos(4 * np.pi * panel["day_of_week"] / 7)
panel["month_sin"]        = np.sin(2 * np.pi * panel["month"] / 12)
panel["month_cos"]        = np.cos(2 * np.pi * panel["month"] / 12)
panel["is_weekend"]       = (panel["day_of_week"] >= 5).astype(int)
panel["is_holiday"]       = panel["timestamp"].dt.date.isin(US_HOLIDAYS.date).astype(int)

# Community-area fixed effects (spatial identity)
dummies = pd.get_dummies(panel["community_area"], prefix="area", drop_first=True).astype(float)
DUMMY_COLS = list(dummies.columns)
panel = pd.concat([panel, dummies], axis=1)

panel = panel.sort_values(["community_area", "timestamp"]).reset_index(drop=True)
print(f"{FREQ}: {len(panel):,} rows ({panel['community_area'].nunique()} areas x "
      f"{panel['timestamp'].nunique():,} windows), {len(DUMMY_COLS)} area dummies, "
      f"zero-demand windows: {(panel['count']==0).mean()*100:.1f}%")

### 2.1 Why a Log-Scale Target

If anything more important here than at 4h: 1h counts are smaller and zero-inflation
heavier.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(panel["count"], bins=60, color=SEQUENTIAL_BLUE)
axes[0].set_title("Raw demand")
axes[0].set_xlabel("Trips / 1h window")
axes[0].set_ylabel("Number of windows")

axes[1].hist(np.log1p(panel["count"]), bins=60, color=SEQUENTIAL_BLUE)
axes[1].set_title("log1p(demand)")
axes[1].set_xlabel("log1p(trips / 1h window)")

area_median_sorted = panel.groupby("community_area")["count"].median().sort_values()
axes[2].bar(range(len(area_median_sorted)), area_median_sorted.to_numpy(), color=SEQUENTIAL_BLUE, width=0.8)
axes[2].set_title("Median demand per area")
axes[2].set_xlabel("Community area (sorted)")
axes[2].set_ylabel("Median trips / 1h window")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("Demand is heavily right-skewed and varies across areas -> log1p target")
plt.tight_layout()
plt.show()

## 3. Feature Set & Shared Helpers

In [ ]:
FEATURE_COLS = [
    "temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "cloud_cover",
    "hour_sin", "hour_cos", "hour_sin2", "hour_cos2",
    "day_of_week_sin", "day_of_week_cos", "day_of_week_sin2", "day_of_week_cos2",
    "month_sin", "month_cos", "is_weekend", "is_holiday",
]
ALL_COLS = FEATURE_COLS + DUMMY_COLS

preprocess = ColumnTransformer([
    ("scale", StandardScaler(), FEATURE_COLS),
    ("area_dummies", "passthrough", DUMMY_COLS),
])

train = panel[panel["timestamp"] < TRAIN_END]
test  = panel[panel["timestamp"] >= TRAIN_END]
y_true = test["count"].to_numpy(dtype=float)
print(f"train={len(train):,} rows   test={len(test):,} rows")


def metrics(y_true, y_pred):
    return {
        "r2":   round(r2_score(y_true, y_pred), 4),
        "mae":  round(mean_absolute_error(y_true, y_pred), 2),
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
    }


# Train-period mean demand per area -- same convention as 03.05's NN notebook (mean, not
# median, unlike AREA_MEDIAN_DEMAND below), so weighted_rmse is directly comparable across
# the two notebooks.
UNIT_WEIGHT = train.groupby("community_area")["count"].mean()
TEST_UNIT_WEIGHTS = test["community_area"].map(UNIT_WEIGHT).fillna(UNIT_WEIGHT.mean()).to_numpy()


def weighted_rmse(y_true, y_pred, weights):
    """Pooled RMSE with each row weighted by its area's train-period mean demand, so errors
    in high-volume areas count more. Complements (not a replacement for) skill_vs_baseline's
    per-area, baseline-relative weighting below -- this one is absolute-scale and pooled,
    like plain RMSE, just demand-weighted instead of uniformly weighted across rows."""
    return float(np.sqrt(np.average((np.asarray(y_true) - np.asarray(y_pred)) ** 2, weights=weights)))


def predict_counts(model, X):
    """Back-transform log-scale predictions to counts, clipped at 0 (and capped pre-expm1
    so an unclipped kernel prediction can't overflow to inf)."""
    y_log = np.clip(model.predict(X), -LOG_PRED_CAP, LOG_PRED_CAP)
    return np.clip(np.expm1(y_log), 0, None)


def per_area_results(test_df, y_pred, group_col="community_area"):
    """One row per spatial unit: MAE/R2/RMSE of the pooled model's predictions for that unit."""
    tmp = pd.DataFrame({
        group_col: test_df[group_col].to_numpy(),
        "y_true": test_df["count"].to_numpy(dtype=float),
        "y_pred": y_pred,
    })
    rows = [
        {group_col: g, **metrics(g_df["y_true"], g_df["y_pred"])}
        for g, g_df in tmp.groupby(group_col)
    ]
    return pd.DataFrame(rows)


def skill_vs_baseline(results_df, baseline_df=None, group_col="community_area"):
    """Per-unit skill score: 1 - MAE_model / MAE_baseline -- comparable across units of
    different scale (MAE rather than RMSE so the baseline, which predicts the group mean,
    gets no built-in advantage under the scoring loss)."""
    if baseline_df is None:
        baseline_df = baseline_results
    base = baseline_df[[group_col, "mae"]].rename(columns={"mae": "mae_base"})
    merged = results_df.merge(base, on=group_col)
    return (1 - merged["mae"] / merged["mae_base"]).rename("skill")


AREA_MEDIAN_DEMAND = panel.groupby("community_area")["count"].median().rename("median_demand")


def weighted_skill_vs_baseline(results_df, baseline_df=None, median_demand=None, group_col="community_area"):
    """Demand-weighted mean skill: each unit's skill weighted by its own median demand --
    the headline metric (fleet is deployed mostly to high-demand areas)."""
    if median_demand is None:
        median_demand = AREA_MEDIAN_DEMAND
    skill = skill_vs_baseline(results_df, baseline_df=baseline_df, group_col=group_col)
    weights = results_df[[group_col]].merge(
        median_demand, left_on=group_col, right_index=True
    )["median_demand"].to_numpy()
    return float(np.average(skill.to_numpy(), weights=weights))


def relative_mae(results_df, median_demand=None, group_col="community_area"):
    """MAE normalized by each unit's own median demand (+1) -- an absolute, baseline-free
    error scale, complementary to skill."""
    if median_demand is None:
        median_demand = AREA_MEDIAN_DEMAND
    demand = results_df[[group_col]].merge(
        median_demand, left_on=group_col, right_index=True
    )["median_demand"].to_numpy()
    return pd.Series(results_df["mae"].to_numpy() / (demand + 1), index=results_df.index, name="relative_mae")


def print_pooled_results(label, pooled_metrics, per_area_df, baseline_df=None, median_demand=None,
                          group_col="community_area", weighted_rmse_value=None):
    w_skill = weighted_skill_vs_baseline(
        per_area_df, baseline_df=baseline_df, median_demand=median_demand, group_col=group_col,
    )
    print(label)
    print(f"  Pooled       : R2={pooled_metrics['r2']:.3f}  MAE={pooled_metrics['mae']:.2f}  "
          f"RMSE={pooled_metrics['rmse']:.2f}  (sanity check only)")
    if weighted_rmse_value is not None:
        print(f"  Weighted RMSE: {weighted_rmse_value:.2f}  (headline)")
    print(f"  Skill        : demand-weighted (headline)={w_skill:+.3f}  "
          f"unweighted median={per_area_df['skill'].median():+.3f}  "
          f"median R2={per_area_df['r2'].median():.3f}  "
          f"(areas with negative skill: {(per_area_df['skill']<0).sum()} of {len(per_area_df)})")

### 3.1 Feature Correlations

Re-run rather than assumed: hour sin/cos now spans 24 values instead of 6 -- confirms
orthogonality still holds exactly.

In [ ]:
corr = train[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap=DIVERGING_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_yticks(range(len(FEATURE_COLS)))
ax.set_xticklabels(FEATURE_COLS, rotation=90, fontsize=8)
ax.set_yticklabels(FEATURE_COLS, fontsize=8)
for i in range(len(FEATURE_COLS)):
    for j in range(len(FEATURE_COLS)):
        val = corr.iloc[i, j]
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=6,
                color="white" if abs(val) > 0.6 else "black")
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
ax.set_title("Feature correlation matrix (train set, 1h)")
plt.tight_layout()
plt.show()

## 4. Profile Baseline

24 hourly buckets instead of 6 -- captures within-day patterns (e.g. exact rush-hour peak)
the 4h baseline blurs together. Expect a stronger reference here, not weaker.

In [ ]:
def profile_baseline(panel, groups, group_col="community_area"):
    results = []
    y_true_all, y_pred_all = [], []
    for g in groups:
        df_g = panel[panel[group_col] == g]
        tr = df_g[df_g["timestamp"] < TRAIN_END]
        te = df_g[df_g["timestamp"] >= TRAIN_END]
        profile = tr.groupby(["hour", "day_of_week", "month"])["count"].mean()
        fallback = tr["count"].mean()
        keys = pd.MultiIndex.from_arrays([te["hour"], te["day_of_week"], te["month"]])
        y_pred = profile.reindex(keys).fillna(fallback).to_numpy()
        y_true_ = te["count"].to_numpy(dtype=float)
        results.append({
            group_col: g,
            "median_demand": float(df_g["count"].median()),
            **metrics(y_true_, y_pred),
        })
        y_true_all.append(y_true_)
        y_pred_all.append(y_pred)
    per_group_df = pd.DataFrame(results)
    y_true_concat, y_pred_concat = np.concatenate(y_true_all), np.concatenate(y_pred_all)
    pooled = metrics(y_true_concat, y_pred_concat)
    return per_group_df, pooled, y_true_concat, y_pred_concat


t0 = time.time()
baseline_results, pm_baseline, baseline_y_true, baseline_y_pred = profile_baseline(panel, ALL_AREAS)
baseline_results["community_area"] = baseline_results["community_area"].astype(int)
baseline_results["relative_mae"] = baseline_results["mae"] / (baseline_results["median_demand"] + 1)
fit_time_base = time.time() - t0

print(f"Profile baseline, all 77 areas ({fit_time_base:.0f}s)")
print(f"  Pooled  : R2={pm_baseline['r2']:.3f}  MAE={pm_baseline['mae']:.2f}  RMSE={pm_baseline['rmse']:.2f}  "
      f"Weighted RMSE={weighted_rmse(y_true, baseline_y_pred, TEST_UNIT_WEIGHTS):.2f}")
print(f"  Per area: median R2={baseline_results['r2'].median():.3f}  "
      f"median MAE={baseline_results['mae'].median():.2f}  "
      f"median RMSE={baseline_results['rmse'].median():.2f}  "
      f"(areas with R2<0: {(baseline_results['r2']<0).sum()} of {len(baseline_results)})")

print("\nPer-area profile-baseline performance (sorted by R2, worst first):")
print(baseline_results.sort_values("r2").reset_index(drop=True).to_string(index=False))

plot_df = baseline_results.sort_values("r2").reset_index(drop=True)
vlim = max(0.5, plot_df["r2"].abs().max())
colors = plt.cm.RdYlGn((plot_df["r2"] + vlim) / (2 * vlim))

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.bar(range(len(plot_df)), plot_df["r2"], color=colors, width=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df["community_area"], rotation=90, fontsize=6)
ax.set_xlabel("Community area (sorted by R2)")
ax.set_ylabel("Test R2")
ax.set_title("Profile baseline -- per-area test R2 (all 77 community areas, 1h)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 4.1 Baseline Performance

Actual vs. predicted counts for the top 3 and bottom 3 community areas (by median demand
in the eval week), same test week and plotting convention used later for model comparisons.

In [ ]:
# Show actual vs predicted counts for the top 3 community areas for a week in the test set
eval_mask = (test["timestamp"] >= EVAL_WEEK_START) & (test["timestamp"] < EVAL_WEEK_END)
test_plot = test[eval_mask].reset_index(drop=True)
test_plot_pred_base = baseline_y_pred[eval_mask.to_numpy()]

# Get the top 3 community areas by median demand in the test set
top_3_areas = test_plot.groupby("community_area")["count"].median().sort_values(ascending=False).head(3).index

for area in top_3_areas:
    mask = (test_plot["community_area"] == area).to_numpy()
    area_val = test_plot[mask]
    area_pred = test_plot_pred_base[mask]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(area_val["timestamp"], area_val["count"], label="actual", color=SEQUENTIAL_BLUE)
    ax.plot(area_val["timestamp"], area_pred, label="predicted", color="#e34948", linestyle="--")
    ax.set_title(f"Baseline Prediction: Community Area {area}")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Trips / 1h window")
    ax.legend(loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
# Show actual vs predicted counts for the bottom 3 community areas for the same test week

bottom_3_areas = test_plot.groupby("community_area")["count"].median().sort_values(ascending=True).head(3).index

for area in bottom_3_areas:
    mask = (test_plot["community_area"] == area).to_numpy()
    area_val = test_plot[mask]
    area_pred = test_plot_pred_base[mask]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(area_val["timestamp"], area_val["count"], label="actual", color=SEQUENTIAL_BLUE)
    ax.plot(area_val["timestamp"], area_pred, label="predicted", color="#e34948", linestyle="--")
    ax.set_title(f"Baseline Prediction: Community Area {area}")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Trips / 1h window")
    ax.legend(loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


## 5. Pooled LinearSVR — No Kernel

Same hyperparameters as `03.01` (`C=1.0, epsilon=0.1`), fit on ~4x the rows -- isolates the
effect of resolution, not tuning. Liblinear scales ~linearly in rows, so expect ~4x `03.01`'s
fit time, not more.

In [ ]:
t0 = time.time()
linear_model = Pipeline([
    ("prep", preprocess),
    ("svm", LinearSVR(C=1.0, epsilon=0.1, max_iter=20000, random_state=SEED)),
])
linear_model.fit(train[ALL_COLS], np.log1p(train["count"]))
y_pred_linear = predict_counts(linear_model, test[ALL_COLS])
pm_linear = metrics(y_true, y_pred_linear)
res_linear = per_area_results(test, y_pred_linear)
res_linear["skill"] = skill_vs_baseline(res_linear)
res_linear["relative_mae"] = relative_mae(res_linear)
fit_time_linear = time.time() - t0

print_pooled_results(f"Pooled LinearSVR ({fit_time_linear:.0f}s)", pm_linear, res_linear,
                     weighted_rmse_value=weighted_rmse(y_true, y_pred_linear, TEST_UNIT_WEIGHTS))

### 5.1 Feature Importance (LinearSVR Coefficients)

Worth comparing against `03.01`'s chart: does finer resolution shift importance toward the
hour harmonics, relative to weather?

In [ ]:
coefs = linear_model.named_steps["svm"].coef_
feature_coefs = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": coefs[:len(FEATURE_COLS)],
    "importance": np.abs(coefs[:len(FEATURE_COLS)]),
}).sort_values("importance")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(feature_coefs["feature"], feature_coefs["importance"], color=SEQUENTIAL_BLUE)
axes[0].set_xlabel("|coefficient| (standardized features)")
axes[0].set_title("LinearSVR feature importance (community area, pooled, 1h)")
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].barh(feature_coefs["feature"], feature_coefs["coefficient"],
             color=[SEQUENTIAL_BLUE if c >= 0 else "#e34948" for c in feature_coefs["coefficient"]])
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("coefficient (sign = direction of effect on log-demand)")
axes[1].set_title("LinearSVR coefficient direction")
axes[1].spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# Community-area fixed effects: which areas get the biggest adjustment vs. the reference area
area_coefs = pd.Series(coefs[len(FEATURE_COLS):], index=DUMMY_COLS).sort_values()
top_bottom = pd.concat([area_coefs.head(5), area_coefs.tail(5)])

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top_bottom.index, top_bottom.to_numpy(),
        color=[SEQUENTIAL_BLUE if c >= 0 else "#e34948" for c in top_bottom])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Area fixed-effect coefficient (log-demand offset vs. reference area 1)")
ax.set_title("Largest area fixed effects (top/bottom 5 of 76)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 5.2 Outcome

Demand-weighted `LinearSVR` dropped: `03.01` found it barely moves skill (-0.75 -> -0.61)
while making most areas worse (48/77 -> 67/77 negative), at ~6x the fit cost -- not worth
repeating at 4x the rows. Plain unweighted `LinearSVR` (full data) is the SVM entry here.

## 6. Kernel Comparison

10,000-row subsample, same as `03.01` -- unlike section 5, this doesn't grow with
resolution, so runtime here is unchanged. Informational only; section 7 shows
subsample-tuned results are unreliable.

In [ ]:
rng = np.random.RandomState(SEED)
sub_idx = rng.choice(train.index, size=10000, replace=False)
train_sub = train.loc[sub_idx]

candidates = {
    "no kernel (LinearSVR)": LinearSVR(C=1.0, epsilon=0.1, max_iter=10000, random_state=SEED),
    "linear (SVR)":          SVR(kernel="linear",  C=100, epsilon=0.1),
    "poly, degree 2":        SVR(kernel="poly",    degree=2, C=100, gamma="scale", epsilon=0.1),
    "rbf":                   SVR(kernel="rbf",     C=100, gamma="scale", epsilon=0.1),
    "sigmoid":               SVR(kernel="sigmoid", C=100, gamma="scale", epsilon=0.1),
}

kernel_rows = []
for name, svm in candidates.items():
    t0 = time.time()
    m = Pipeline([("prep", preprocess), ("svm", svm)])
    m.fit(train_sub[ALL_COLS], np.log1p(train_sub["count"]))
    y_pred = predict_counts(m, test[ALL_COLS])
    kernel_rows.append({"kernel": name, **metrics(y_true, y_pred), "fit_seconds": round(time.time() - t0, 1)})

kernel_df = pd.DataFrame(kernel_rows).sort_values("r2", ascending=False)
print("Kernel comparison -- 10,000-row train subsample, full test set, pooled metrics:")
print(kernel_df.to_string(index=False))

Dummies never scaled, same as `03.01` (see its section 6 for why scaling flips which
kernel wins).

## 7. Hyperparameter Tuning — GridSearchCV

Same approach as `03.01`: `GridSearchCV` + `KFold(3, shuffled)` on poly (section 6's naive
winner), same 10,000-row budget, a different draw.

In [ ]:
# KFold, not TimeSeriesSplit: no lagged/historical-demand features means no autoregressive
# leakage risk, so a chronological split isn't required -- and shuffled folds give each fold
# an even mix of seasons instead of the growing-window bias TimeSeriesSplit would introduce.
kfold = KFold(n_splits=3, shuffle=True, random_state=SEED)
rng2 = np.random.RandomState(SEED)
sub_idx2 = rng2.choice(train.index, size=10000, replace=False)
train_sub2 = train.loc[sub_idx2]

poly_param_grid = {"svm__C": [1, 10, 100], "svm__degree": [2, 3], "svm__gamma": ["scale"]}
t0 = time.time()
gs_poly = GridSearchCV(
    Pipeline([("prep", preprocess), ("svm", SVR(kernel="poly", epsilon=0.1))]),
    poly_param_grid, cv=kfold, scoring="r2", n_jobs=-1,
)
gs_poly.fit(train_sub2[FEATURE_COLS + DUMMY_COLS], np.log1p(train_sub2["count"]))
y_pred_poly = predict_counts(gs_poly, test[ALL_COLS])
pm_poly = metrics(y_true, y_pred_poly)
res_poly = per_area_results(test, y_pred_poly)
res_poly["skill"] = skill_vs_baseline(res_poly)
res_poly["relative_mae"] = relative_mae(res_poly)
fit_time_poly = time.time() - t0

print(f"Tuned poly ({fit_time_poly:.0f}s): best_params={gs_poly.best_params_}  "
      f"best_cv_r2={gs_poly.best_score_:.3f}")
print_pooled_results("", pm_poly, res_poly,
                     weighted_rmse_value=weighted_rmse(y_true, y_pred_poly, TEST_UNIT_WEIGHTS))

rbf_param_grid = {"svm__C": [10, 100], "svm__gamma": ["scale", 0.01]}
t0 = time.time()
gs_rbf = GridSearchCV(
    Pipeline([("prep", preprocess), ("svm", SVR(kernel="rbf", epsilon=0.1))]),
    rbf_param_grid, cv=kfold, scoring="r2", n_jobs=-1,
)
gs_rbf.fit(train_sub2[FEATURE_COLS + DUMMY_COLS], np.log1p(train_sub2["count"]))
y_pred_rbf = predict_counts(gs_rbf, test[ALL_COLS])
pm_rbf = metrics(y_true, y_pred_rbf)
res_rbf = per_area_results(test, y_pred_rbf)
res_rbf["skill"] = skill_vs_baseline(res_rbf)
res_rbf["relative_mae"] = relative_mae(res_rbf)
fit_time_rbf = time.time() - t0

print(f"\nTuned RBF ({fit_time_rbf:.0f}s): best_params={gs_rbf.best_params_}  "
      f"best_cv_r2={gs_rbf.best_score_:.3f}")
print_pooled_results("", pm_rbf, res_rbf,
                     weighted_rmse_value=weighted_rmse(y_true, y_pred_rbf, TEST_UNIT_WEIGHTS))

*(Compare against `03.01`'s finding that subsample-tuning is high-variance: tuned poly there
scored `best_cv_r2=0.088` in CV but `R2=-0.032` on the real test set.)*

## 8. Results — Model Comparison

- Four candidates: profile baseline, `LinearSVR` (unweighted), tuned RBF, tuned poly --
  demand-weighted `LinearSVR` dropped per section 5.2's decision.

In [ ]:
comparison_df = pd.DataFrame([
    {"Model": "Profile baseline", "Demand-weighted skill": 0.000, "Median skill": 0.000,
     "Weighted RMSE": weighted_rmse(y_true, baseline_y_pred, TEST_UNIT_WEIGHTS),
     "Median relative MAE": baseline_results["relative_mae"].median(),
     "Pooled R2": pm_baseline["r2"], "Median R2/area": baseline_results["r2"].median()},
    {"Model": "LinearSVR (full data, no kernel)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_linear), "Median skill": res_linear["skill"].median(),
     "Weighted RMSE": weighted_rmse(y_true, y_pred_linear, TEST_UNIT_WEIGHTS),
     "Median relative MAE": res_linear["relative_mae"].median(),
     "Pooled R2": pm_linear["r2"], "Median R2/area": res_linear["r2"].median()},
    {"Model": "RBF, tuned (10k subsample)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_rbf), "Median skill": res_rbf["skill"].median(),
     "Weighted RMSE": weighted_rmse(y_true, y_pred_rbf, TEST_UNIT_WEIGHTS),
     "Median relative MAE": res_rbf["relative_mae"].median(),
     "Pooled R2": pm_rbf["r2"], "Median R2/area": res_rbf["r2"].median()},
    {"Model": "Poly, tuned (10k subsample)",
     "Demand-weighted skill": weighted_skill_vs_baseline(res_poly), "Median skill": res_poly["skill"].median(),
     "Weighted RMSE": weighted_rmse(y_true, y_pred_poly, TEST_UNIT_WEIGHTS),
     "Median relative MAE": res_poly["relative_mae"].median(),
     "Pooled R2": pm_poly["r2"], "Median R2/area": res_poly["r2"].median()},
]).round(3)
print("Two headline metrics: demand-weighted skill (relative to baseline, comparable across")
print("resolutions) and weighted RMSE (absolute trips/window, comparable against the NN")
print("notebooks). Unweighted median skill and relative MAE are supporting detail.")
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
chart_df = comparison_df.set_index("Model")["Demand-weighted skill"]
colors_cmp = [SEQUENTIAL_BLUE if v >= 0 else "#e34948" for v in chart_df]
axes[0].barh(chart_df.index, chart_df.to_numpy(), color=colors_cmp)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Demand-weighted skill vs. profile baseline")
axes[0].set_title("Skill (headline 1/2)")
axes[0].spines[["top", "right"]].set_visible(False)

wrmse_df = comparison_df.set_index("Model")["Weighted RMSE"]
axes[1].barh(wrmse_df.index, wrmse_df.to_numpy(), color=SEQUENTIAL_BLUE)
axes[1].set_xlabel("Weighted RMSE (trips / 1h window)")
axes[1].set_title("Weighted RMSE (headline 2/2)")
axes[1].spines[["top", "right"]].set_visible(False)

rmae_df = comparison_df.set_index("Model")["Median relative MAE"]
axes[2].barh(rmae_df.index, rmae_df.to_numpy(), color=SEQUENTIAL_BLUE)
axes[2].set_xlabel("Median relative MAE (MAE / median demand)")
axes[2].set_title("Relative MAE (supporting detail)")
axes[2].spines[["top", "right"]].set_visible(False)
plt.suptitle("Model comparison -- community area, 1h")
plt.tight_layout()
plt.show()

### 8.0 Actual vs. Predicted (Test Set)

- Same four models as the comparison table above, same test set, same axes -- a tighter
  cluster along the dashed diagonal means better predictions.
- Plotted on `log1p` scale (a 5,000-point random sample, for a readable scatter).

In [ ]:
models_to_plot = [
    ("Profile baseline", baseline_y_pred),
    ("LinearSVR", y_pred_linear),
    ("RBF, tuned", y_pred_rbf),
    ("Poly, tuned", y_pred_poly),
]

rng_viz = np.random.RandomState(0)
sample_n = min(5000, len(y_true))
sample_idx = rng_viz.choice(len(y_true), size=sample_n, replace=False)
log_true = np.log1p(y_true)
lims = [0, log_true.max()]

fig, axes = plt.subplots(1, len(models_to_plot), figsize=(18, 4.5), sharex=True, sharey=True)
for ax, (name, y_pred) in zip(axes, models_to_plot):
    log_pred = np.log1p(y_pred)
    ax.scatter(log_true[sample_idx], log_pred[sample_idx], s=4, alpha=0.25, color=SEQUENTIAL_BLUE)
    ax.plot(lims, lims, color="#e34948", linewidth=1, linestyle="--")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("log1p(actual)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("log1p(predicted)")
plt.suptitle(f"Actual vs. predicted demand -- test set, 1h, {sample_n:,}-point sample "
             f"(dashed = perfect prediction)")
plt.tight_layout()
plt.show()

### 8.1 Per-Area Performance

- **Two complementary scores**, both comparable across areas of very different demand:
  **skill** (relative to the profile baseline) and **relative MAE** (`mae / (median_demand
  + 1)`, absolute, no baseline).
- Shown for the plain **unweighted LinearSVR**.

In [ ]:
res_linear_by_skill = res_linear.merge(
    AREA_MEDIAN_DEMAND, left_on="community_area", right_index=True
).sort_values("skill").reset_index(drop=True)
colors_skill = [SEQUENTIAL_BLUE if v >= 0 else "#e34948" for v in res_linear_by_skill["skill"]]

res_linear_by_rmae = res_linear.merge(
    AREA_MEDIAN_DEMAND, left_on="community_area", right_index=True
).sort_values("relative_mae", ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
axes[0].bar(range(len(res_linear_by_skill)), res_linear_by_skill["skill"], color=colors_skill, width=0.8)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xticks(range(len(res_linear_by_skill)))
axes[0].set_xticklabels(res_linear_by_skill["community_area"], rotation=90, fontsize=6)
axes[0].set_xlabel("Community area (sorted by skill)")
axes[0].set_ylabel("Skill vs. profile baseline")
axes[0].set_title("Skill (relative to baseline)")

axes[1].bar(range(len(res_linear_by_rmae)), res_linear_by_rmae["relative_mae"], color=SEQUENTIAL_BLUE, width=0.8)
axes[1].set_xticks(range(len(res_linear_by_rmae)))
axes[1].set_xticklabels(res_linear_by_rmae["community_area"], rotation=90, fontsize=6)
axes[1].set_xlabel("Community area (sorted by relative MAE, worst first)")
axes[1].set_ylabel("MAE / median demand")
axes[1].set_title("Relative MAE (absolute, no baseline)")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.suptitle("LinearSVR (unweighted) -- per-area performance, two views (1h)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(res_linear_by_skill["median_demand"] + 1, res_linear_by_skill["skill"], color=colors_skill, s=25)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xscale("log")
axes[0].set_xlabel("Median demand per area (log scale, +1)")
axes[0].set_ylabel("Skill vs. profile baseline")
axes[0].set_title("Skill vs. area demand")

axes[1].scatter(res_linear_by_rmae["median_demand"] + 1, res_linear_by_rmae["relative_mae"], color=SEQUENTIAL_BLUE, s=25)
axes[1].set_xscale("log")
axes[1].set_xlabel("Median demand per area (log scale, +1)")
axes[1].set_ylabel("MAE / median demand")
axes[1].set_title("Relative MAE vs. area demand")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

Per-area skill map for `LinearSVR` (plain, unweighted, full pooled data, 1h) -- no areas
excluded.

In [ ]:
community_areas_gdf = gpd.read_file(COMMUNITY_AREAS_PATH)
community_areas_gdf["area_numbe"] = community_areas_gdf["area_numbe"].astype(int)

gdf_skill = community_areas_gdf.merge(
    res_linear[["community_area", "skill"]],
    left_on="area_numbe", right_on="community_area", how="left",
).to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(9, 9))
gdf_skill.plot(ax=ax, column="skill", cmap="RdYlGn", vmin=-0.5, vmax=0.5, legend=True,
               legend_kwds={"label": "Test skill vs. profile baseline (pooled LinearSVR, 1h)", "shrink": 0.6},
               edgecolor="white", linewidth=0.4)
ctx.add_basemap(ax, crs=gdf_skill.crs, source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.35)
ax.set_title("Pooled LinearSVR -- skill vs. baseline per community area (all 77 areas, 1h)")
ax.axis("off")
plt.tight_layout()
plt.show()

### 8.2 Top 3 / Bottom 3 Areas -- All Models

Actual vs. all four models for the top-3 and bottom-3 areas (by median demand), same
eval week used for the baseline-only view in section 4.1 -- now with every model overlaid
so the comparison table's numbers are visible as shapes, not just numbers.

In [ ]:
# Same eval_mask/test_plot as section 4.1 -- slice every other model's predictions the same way.
test_plot_pred_linear = y_pred_linear[eval_mask.to_numpy()]
test_plot_pred_rbf    = y_pred_rbf[eval_mask.to_numpy()]
test_plot_pred_poly   = y_pred_poly[eval_mask.to_numpy()]

models_to_plot_ts = [
    ("baseline", test_plot_pred_base),
    ("LinearSVR", test_plot_pred_linear),
    ("RBF, tuned", test_plot_pred_rbf),
    ("Poly, tuned", test_plot_pred_poly),
]


def plot_area_all_models(areas, title_prefix):
    for area in areas:
        mask = (test_plot["community_area"] == area).to_numpy()
        area_val = test_plot[mask]

        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(area_val["timestamp"], area_val["count"], label="actual", color=CATEGORICAL_COLORS[0])
        for i, (name, y_pred) in enumerate(models_to_plot_ts):
            ax.plot(area_val["timestamp"], y_pred[mask], label=name,
                    color=CATEGORICAL_COLORS[i + 1], linestyle="--")
        ax.set_title(f"{title_prefix}: Community Area {area}")
        ax.set_xlabel("Timestamp")
        ax.set_ylabel("Trips / 1h window")
        ax.legend(loc="upper right", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        plt.show()


plot_area_all_models(top_3_areas, "Top-3 area")


In [ ]:
plot_area_all_models(bottom_3_areas, "Bottom-3 area")


## 9. Export Results

Append this run's four pooled models from section 8 to the shared results CSVs: a per-model
summary (`model_results_summary.csv`) and a per-area detail (`model_results_by_area.csv`).
Reruns overwrite this notebook's own prior rows (matched on model + notebook + resolution)
rather than duplicating them.

In [ ]:
RESULTS_DIR = pathlib.Path("../../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = RESULTS_DIR / "model_results_summary.csv"
DETAIL_CSV  = RESULTS_DIR / "model_results_by_area.csv"
NOTEBOOK_NAME = "03.02_prediction_svm_CA_1h.ipynb"


def upsert_csv(df_new, path, key_cols):
    """Append df_new to the CSV at `path`, replacing any existing rows with the same key_cols
    (so reruns update in place instead of duplicating) -- creates the file if it doesn't exist."""
    if path.exists():
        df_old = pd.read_csv(path)
        old_keys = df_old[key_cols].apply(tuple, axis=1)
        new_keys = set(df_new[key_cols].apply(tuple, axis=1))
        df_out = pd.concat([df_old[~old_keys.isin(new_keys)], df_new], ignore_index=True)
    else:
        df_out = df_new
    df_out.to_csv(path, index=False)
    return df_out


run_timestamp = pd.Timestamp.now().isoformat(timespec="seconds")

# (label, pooled metrics, per-area results, y_true used, y_pred, weights used, fit time,
#  demand-weighted skill, median skill, n_units) -- all four models here are pooled and cover
# all 77 areas, so they all share the full-test y_true/TEST_UNIT_WEIGHTS.
svm_models = [
    ("Profile baseline", pm_baseline, baseline_results, y_true, baseline_y_pred, TEST_UNIT_WEIGHTS,
     fit_time_base, 0.0, 0.0, len(ALL_AREAS)),
    ("LinearSVR (full data, no kernel)", pm_linear, res_linear, y_true, y_pred_linear, TEST_UNIT_WEIGHTS,
     fit_time_linear, weighted_skill_vs_baseline(res_linear), res_linear["skill"].median(), len(ALL_AREAS)),
    ("RBF, tuned (10k subsample)", pm_rbf, res_rbf, y_true, y_pred_rbf, TEST_UNIT_WEIGHTS,
     fit_time_rbf, weighted_skill_vs_baseline(res_rbf), res_rbf["skill"].median(), len(ALL_AREAS)),
    ("Poly, tuned (10k subsample)", pm_poly, res_poly, y_true, y_pred_poly, TEST_UNIT_WEIGHTS,
     fit_time_poly, weighted_skill_vs_baseline(res_poly), res_poly["skill"].median(), len(ALL_AREAS)),
]

summary_rows, detail_rows = [], []
for model_name, pm, res_df, y_true_m, y_pred_m, weights_m, fit_time, skill_w, skill_med, n_units in svm_models:
    summary_rows.append({
        "model_name": model_name,
        "notebook": NOTEBOOK_NAME,
        "spatial_resolution": "community_area",
        "temporal_resolution": FREQ,
        "n_units": n_units,
        "train_start": DATA_START.date().isoformat(),
        "train_end": (TRAIN_END - pd.Timedelta(days=1)).date().isoformat(),
        "test_start": TRAIN_END.date().isoformat(),
        "test_end": (DATA_END_EXCL - pd.Timedelta(days=1)).date().isoformat(),
        "rmse": pm["rmse"],
        "mae": pm["mae"],
        "r2": pm["r2"],
        "weighted_rmse": weighted_rmse(y_true_m, y_pred_m, weights_m),
        "skill_demand_weighted": round(float(skill_w), 4),
        "skill_median": round(float(skill_med), 4),
        "fit_time_s": round(fit_time, 2),
        "last_run_at": run_timestamp,
    })
    for _, row in res_df.iterrows():
        detail_rows.append({
            "model_name": model_name,
            "notebook": NOTEBOOK_NAME,
            "spatial_resolution": "community_area",
            "temporal_resolution": FREQ,
            "unit_id": int(row["community_area"]),
            "median_demand": AREA_MEDIAN_DEMAND.get(row["community_area"]),
            "rmse": row["rmse"],
            "mae": row["mae"],
            "r2": row["r2"],
            "skill": row.get("skill", 0.0),
            "relative_mae": row["relative_mae"],
            "last_run_at": run_timestamp,
        })

summary_df = pd.DataFrame(summary_rows)
detail_df = pd.DataFrame(detail_rows)

upsert_csv(summary_df, SUMMARY_CSV,
           key_cols=["model_name", "notebook", "spatial_resolution", "temporal_resolution"])
upsert_csv(detail_df, DETAIL_CSV,
           key_cols=["model_name", "notebook", "spatial_resolution", "temporal_resolution", "unit_id"])

print(f"Wrote {len(summary_df)} rows to {SUMMARY_CSV}")
print(f"Wrote {len(detail_df)} rows to {DETAIL_CSV}")
summary_df

## 10. Shortfalls & Honest Limitations

- **Pooled log-scale regression struggles with extreme-demand areas**, same `expm1`
  back-transform concern as `03.01`.
- **Neither demand tiering nor per-area models are covered here**: reproducing `03.01`'s
  per-area breakdown at 1h crashed the notebook on available hardware (parallelized
  per-area `GridSearchCV` fan-out on the larger 1h panel) -- see `03.01` for that comparison.
- **Fixed effects assume every unit is seen in training**; no rolling-origin backtest yet --
  same as `03.01`.

## 11. Outlook

- **Census tract + 1h combined**: not attempted -- `03.03` (tract) and this notebook each
  explore one scaling axis in isolation; combining both multiplies both problems (row count,
  per-unit tuning cost).
- Results feed the SVM section of the report (`sections/04-prediction_svm.qmd`), alongside
  `03.01` and `03.03`.